In [10]:
# Importowanie bibliotek
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import librosa
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

In [11]:
# Konfiguracja urządzenia
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

# Klasyfikator audio
class EnhancedAudioClassifier(nn.Module):
    def __init__(self, input_size=20, hidden_size=256, num_layers=3, output_size=3):
        super(EnhancedAudioClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, 
                            batch_first=True, dropout=0.3)
        self.batch_norm = nn.BatchNorm1d(hidden_size)
        self.fc1 = nn.Linear(hidden_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, output_size)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = x.permute(0, 2, 1)  # Swap dimensions for LSTM
        _, (hn, _) = self.lstm(x)
        x = self.batch_norm(hn[-1])  # Take the last hidden state and apply normalization
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

Device: cuda


In [12]:
# Dataset audio
class AudioDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        mfcc = self.data[idx]
        label = self.labels[idx]
        return torch.tensor(mfcc).float(), torch.tensor(label).long()


In [13]:
# Funkcja ładowania danych z augmentacją
def load_data_with_augmentation(data_dirs, labels, sr=48000, chunk_size=48000, augment=True):
    data = []
    label_list = []
    for dir_path, label in zip(data_dirs, labels):
        for file in os.listdir(dir_path):
            file_path = os.path.join(dir_path, file)
            y, _ = librosa.load(file_path, sr=sr, mono=True)
            for i in range(0, len(y), chunk_size):
                chunk = y[i:i + chunk_size]
                if len(chunk) == chunk_size:
                    mfcc = librosa.feature.mfcc(y=chunk, sr=sr)
                    if augment:
                        mfcc = augment_features(mfcc)
                    data.append(mfcc)
                    label_list.append(label)
    return np.array(data), np.array(label_list)


In [14]:
# Funkcja augmentacji
def augment_features(mfcc):
    # Add noise
    noise = np.random.normal(0, 0.01, mfcc.shape)
    return mfcc + noise


In [15]:
# Konfiguracja danych
data_dirs = ['../data1/exhale', '../data1/inhale', '../data1/silence']
labels = [0, 1, 2]


In [16]:
# Ładowanie danych
print("Loading and augmenting data...")
data, labels = load_data_with_augmentation(data_dirs, labels)
print(len(data))
train_data, val_data, train_labels, val_labels = train_test_split(data, labels, test_size=0.2, random_state=42)


Loading and augmenting data...
1378


In [17]:
# Tworzenie zestawów danych i DataLoaderów
train_dataset = AudioDataset(train_data, train_labels)
val_dataset = AudioDataset(val_data, val_labels)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)


In [ ]:
# Trening modelu
model = EnhancedAudioClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

NUM_EPOCHS = 50
best_val_f1 = 0.0
for epoch in range(NUM_EPOCHS):
    print(f"\n{'='*30}\nEpoch {epoch + 1}/{NUM_EPOCHS}\n{'='*30}")
    
    # Trening
    model.train()
    running_loss = 0.0
    train_preds, train_labels_all = [], []
    for batch_idx, (inputs, labels) in enumerate(tqdm(train_loader, desc="Training")):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        train_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
        train_labels_all.extend(labels.cpu().numpy())

        # Debug info for every 10 batches
        if batch_idx % 10 == 0:
            print(f"[Train Batch {batch_idx}/{len(train_loader)}] Loss: {loss.item():.4f}")

    train_f1 = f1_score(train_labels_all, train_preds, average='macro')
    print(f"Train Loss: {running_loss / len(train_loader):.4f}, Train F1: {train_f1:.4f}")

    # Walidacja
    model.eval()
    val_loss = 0.0
    val_preds, val_labels_all = [], []
    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(tqdm(val_loader, desc="Validation")):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            val_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            val_labels_all.extend(labels.cpu().numpy())

            # Debug info for every 10 batches
            if batch_idx % 10 == 0:
                print(f"[Val Batch {batch_idx}/{len(val_loader)}] Loss: {loss.item():.4f}")

    val_f1 = f1_score(val_labels_all, val_preds, average='macro')
    print(f"Val Loss: {val_loss / len(val_loader):.4f}, Val F1: {val_f1:.4f}")

    scheduler.step()

    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'best_audio_model.pth')
        print(f"Model saved with best F1 score: {best_val_f1:.4f}!")

    # Debugging info after each epoch
    print(f"Epoch {epoch + 1} completed. Current best F1: {best_val_f1:.4f}")

print("Training complete.")


Epoch 1/50


Training:   0%|          | 0/35 [00:00<?, ?it/s]